In [ ]:
 # ===============================
# IMPORT LIBRARIES
# ===============================
import random
import string
import pandas as pd
import math

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ===============================
# PASSWORD GENERATION
# ===============================
def generate_password():
    chars = string.ascii_letters + string.digits + "!@#$%^&*"
    length = random.randint(5, 12)
    return ''.join(random.choice(chars) for _ in range(length))

# ===============================
# LABEL FUNCTION (SUPERVISED LEARNING)
# ===============================
def label_password(pwd):
    score = 0

    if len(pwd) >= 8:
        score += 1
    if any(c.isupper() for c in pwd):
        score += 1
    if any(c.isdigit() for c in pwd):
        score += 1
    if any(c in "!@#$%^&*" for c in pwd):
        score += 1

    if score <= 1:
        return "weak"
    elif score == 2:
        return "medium"
    else:
        return "strong"

# ===============================
# FEATURE EXTRACTION
# ===============================
def extract_features(pwd):
    return [
        len(pwd),
        sum(c.isdigit() for c in pwd),
        sum(c.isupper() for c in pwd),
        sum(c.islower() for c in pwd),
        sum(c in "!@#$%^&*" for c in pwd)
    ]

# ===============================
# GENERATE DATASET
# ===============================
data = []

for _ in range(2000):
    pwd = generate_password()
    features = extract_features(pwd)
    label = label_password(pwd)
    data.append(features + [label])

df = pd.DataFrame(data, columns=[
    "length", "digits", "upper", "lower", "special", "strength"
])

print("Dataset Sample:\n")
print(df.head())

# ===============================
# SPLIT DATA
# ===============================
X = df.iloc[:, :-1]
y = df["strength"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ===============================
# TRAIN MODELS
# ===============================
lr_model = LogisticRegression(max_iter=200)
lr_model.fit(X_train, y_train)

dt_model = DecisionTreeClassifier()
dt_model.fit(X_train, y_train)

# ===============================
# EVALUATE MODELS
# ===============================
lr_pred = lr_model.predict(X_test)
dt_pred = dt_model.predict(X_test)

lr_acc = accuracy_score(y_test, lr_pred)
dt_acc = accuracy_score(y_test, dt_pred)

print("\nModel Accuracy:")
print("Logistic Regression:", lr_acc)
print("Decision Tree:", dt_acc)

# ===============================
# CLASSIFICATION REPORT
# ===============================
print("\nLogistic Regression Report:\n")
print(classification_report(y_test, lr_pred))

print("\nDecision Tree Report:\n")
print(classification_report(y_test, dt_pred))

# ===============================
# CONFUSION MATRIX
# ===============================
print("\nDecision Tree Confusion Matrix:\n")
print(confusion_matrix(y_test, dt_pred))

# ===============================
# SELECT BEST MODEL
# ===============================
if lr_acc > dt_acc:
    best_model = lr_model
    model_name = "Logistic Regression"
else:
    best_model = dt_model
    model_name = "Decision Tree"

print("\nBest Model Selected:", model_name)

# ===============================
# ENTROPY FUNCTION
# ===============================
def calculate_entropy(password):
    char_set = 0

    if any(c.islower() for c in password):
        char_set += 26
    if any(c.isupper() for c in password):
        char_set += 26
    if any(c.isdigit() for c in password):
        char_set += 10
    if any(c in "!@#$%^&*" for c in password):
        char_set += 32

    if char_set == 0:
        return 0

    return len(password) * math.log2(char_set)

# ===============================
# CRACK TIME FUNCTION (NEW)
# ===============================
def crack_time(password):
    guesses_per_second = 1e6  # attacker speed

    entropy = calculate_entropy(password)
    seconds = 2 ** entropy / guesses_per_second

    if seconds < 60:
        return f"{seconds:.2f} seconds"
    elif seconds < 3600:
        return f"{seconds/60:.2f} minutes"
    elif seconds < 86400:
        return f"{seconds/3600:.2f} hours"
    elif seconds < 31536000:
        return f"{seconds/86400:.2f} days"
    else:
        return f"{seconds/31536000:.2f} years"

# ===============================
# FINAL PREDICTION FUNCTION
# ===============================
def predict_password(password):
    features = extract_features(password)
    prediction = best_model.predict([features])[0]
    entropy = calculate_entropy(password)
    time_to_crack = crack_time(password)

    return prediction, entropy, time_to_crack

# ===============================
# TEST WITH USER INPUT
# ===============================
while True:
    user_pwd = input("\nEnter a password (or type 'exit'): ")

    if user_pwd.lower() == 'exit':
        break

    strength, entropy, crack = predict_password(user_pwd)

    print(f"Predicted Strength: {strength}")
    print(f"Entropy: {entropy:.2f}")
    print(f"Estimated Crack Time: {crack}")

Dataset Sample:

   length  digits  upper  lower  special strength
0       8       0      2      4        2   strong
1       6       0      3      3        0     weak
2       9       1      3      4        1   strong
3       9       1      4      3        1   strong
4       6       0      3      3        0     weak

Model Accuracy:
Logistic Regression: 0.825
Decision Tree: 1.0

Logistic Regression Report:

              precision    recall  f1-score   support

      medium       0.73      0.62      0.67       112
      strong       0.86      0.91      0.88       258
        weak       0.84      0.87      0.85        30

    accuracy                           0.82       400
   macro avg       0.81      0.80      0.80       400
weighted avg       0.82      0.82      0.82       400


Decision Tree Report:

              precision    recall  f1-score   support

      medium       1.00      1.00      1.00       112
      strong       1.00      1.00      1.00       258
        weak       1.0

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(


Predicted Strength: weak
Entropy: 13.29
Estimated Crack Time: 0.01 seconds


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(


Predicted Strength: weak
Entropy: 13.29
Estimated Crack Time: 0.01 seconds


...

Predicted Strength: medium
Entropy: 47.00
Estimated Crack Time: 4.48 years


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(


Predicted Strength: weak
Entropy: 18.80
Estimated Crack Time: 0.46 seconds


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(


Predicted Strength: weak
Entropy: 14.10
Estimated Crack Time: 0.02 seconds

Enter a password (or type 'exit'): exit
